# Topic 3 — Statistics
### 3-hour seminar

**Suggested pacing (3h):**
- 0:00–0:15 — Readings/objectives
- 0:15–1:15 — Part 1: descriptive & inferential statistics on simulated data
- 1:15–1:25 — Break
- 1:25–2:45 — Part 2: hypothesis testing on a real Kaggle dataset (Students Performance)
- 2:45–3:00 — Wrap-up questions


## Learning objectives
- Compute and interpret descriptive statistics (mean, median, variance, skew)
- Explain and simulate the Central Limit Theorem
- Run and interpret a t-test, ANOVA, and chi-square test
- Compute and interpret a confidence interval


## Required readings
- Downey, *Think Stats* (free) — Ch. 7 & 9: https://greenteapress.com/wp/think-stats-2e/
- scipy.stats documentation: https://docs.scipy.org/doc/scipy/reference/stats.html
- DA Notebook 5 (correlation tests)

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(7)


---
## Part 1 — Simulated data

We'll simulate two "treatment groups" to practice hypothesis testing before touching real data.


In [ ]:
group_a = np.random.normal(loc=50, scale=10, size=120)   # control
group_b = np.random.normal(loc=54, scale=10, size=120)   # treatment
df = pd.DataFrame({"score": np.concatenate([group_a, group_b]),
                    "group": ["A"]*120 + ["B"]*120})
df.head()


### Exercise 1.1 — Descriptive statistics
For each group, compute mean, median, standard deviation, and skewness
(`scipy.stats.skew`). Plot overlapping histograms of the two groups.


In [ ]:
# Описательные показатели для каждой группы
for group_name, group_data in df.groupby("group"):
    scores = group_data["score"]
    print(f"Группа {group_name}")
    print(f"  Среднее: {scores.mean():.2f}")
    print(f"  Медиана: {scores.median():.2f}")
    print(f"  Стандартное отклонение: {scores.std():.2f}")
    print(f"  Асимметрия: {stats.skew(scores):.3f}")
    print()

# Гистограммы с наложением
plt.figure(figsize=(8, 5))
plt.hist(group_a, bins=20, alpha=0.55, label="Group A")
plt.hist(group_b, bins=20, alpha=0.55, label="Group B")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Score distributions by group")
plt.legend()
plt.show()

### Exercise 1.2 — Central Limit Theorem
1. Draw 1000 samples of size 30 from `group_a`'s *population distribution*
   (i.e. `np.random.normal(50, 10, 30)`), computing the mean of each sample
2. Plot a histogram of these 1000 sample means
3. Compare its shape and spread to the original population distribution


In [ ]:
# 1000 выборок размера 30 из исходного распределения N(50, 10)
sample_means = [np.mean(np.random.normal(50, 10, 30)) for _ in range(1000)]

plt.figure(figsize=(8, 5))
plt.hist(sample_means, bins=30, edgecolor="white")
plt.xlabel("Sample mean")
plt.ylabel("Frequency")
plt.title("Sampling distribution of the mean (n=30)")
plt.show()

print(f"Среднее выборочных средних: {np.mean(sample_means):.2f}")
print(f"Стандартное отклонение выборочных средних: {np.std(sample_means, ddof=1):.2f}")
print(f"Теоретическая стандартная ошибка: {10 / np.sqrt(30):.2f}")

### Exercise 1.3 — Hypothesis test
Test whether group B's mean is significantly higher than group A's using an
independent two-sample t-test (`stats.ttest_ind`). State your null hypothesis,
alternative hypothesis, and conclusion at alpha = 0.05.


In [ ]:
# H0: среднее группы B не выше среднего группы A (μB <= μA)
# H1: среднее группы B выше среднего группы A (μB > μA)
t_stat, p_two_sided = stats.ttest_ind(group_b, group_a, equal_var=True)
# Перевод двустороннего p-value в односторонний для направленной альтернативы.
# Здесь t_stat положителен, поэтому p_one_sided = p_two_sided / 2.
p_one_sided = p_two_sided / 2 if t_stat > 0 else 1 - p_two_sided / 2
print(f"t-statistic: {t_stat:.3f}")
print(f"one-sided p-value: {p_one_sided:.6f}")
alpha = 0.05
if p_one_sided < alpha:
    print("Вывод: отвергаем H0; есть статистические основания считать, что среднее B выше среднего A.")
else:
    print("Вывод: недостаточно оснований утверждать, что среднее B выше среднего A.")

**H₀:** средний результат группы B не выше результата группы A (μB ≤ μA).  
**H₁:** средний результат группы B выше результата группы A (μB > μA).  
Так как одностороннее p-value меньше 0.05, отвергаем H₀: в этой симуляции результаты группы B статистически значимо выше. Это вывод о смоделированных данных, а не доказательство причинного эффекта в реальном исследовании.

### Exercise 1.4 — Confidence interval
Compute a 95% confidence interval for the mean of `group_b` using
`stats.t.interval` (or the normal approximation).


In [ ]:
# 95% доверительный интервал для среднего группы B
n = len(group_b)
mean_b = np.mean(group_b)
sem_b = stats.sem(group_b)
ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean_b, scale=sem_b)
print(f"Среднее группы B: {mean_b:.2f}")
print(f"95% доверительный интервал: [{ci_low:.2f}, {ci_high:.2f}]")

---
## Part 2 — Real data: Students Performance in Exams (Kaggle)

**Get the data:** https://www.kaggle.com/datasets/spscientist/students-performance-in-exams
Download `StudentsPerformance.csv` and place it next to this notebook.


In [ ]:
from pathlib import Path
DATA_PATH = "StudentsPerformance.csv"  # положите Kaggle CSV рядом с ноутбуком
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        "Не найден StudentsPerformance.csv. Скачайте файл StudentsPerformance.csv "
        "из Kaggle и положите его в ту же папку, что и ноутбук."
    )
students = pd.read_csv(DATA_PATH)
students.head()

### Exercise 2.1 — Descriptive overview
Compute mean/median/std of `math score`, `reading score`, `writing score`,
overall and split by `gender`.


In [ ]:
score_columns = ["math score", "reading score", "writing score"]
print("Общая описательная статистика:")
print(students[score_columns].agg(["mean", "median", "std"]).T.round(2))
print("\nПоказатели в разбивке по gender:")
print(students.groupby("gender")[score_columns].agg(["mean", "median", "std"]).round(2))

### Exercise 2.2 — t-test: does test preparation help?
Compare `math score` between students who completed `test preparation course`
vs. those who didn't. Run a t-test and interpret the result.


In [ ]:
prepared = students.loc[students["test preparation course"] == "completed", "math score"]
not_prepared = students.loc[students["test preparation course"] == "none", "math score"]
t_stat, p_value = stats.ttest_ind(prepared, not_prepared, equal_var=False)
print(f"Среднее, курс завершён: {prepared.mean():.2f}")
print(f"Среднее, курс не завершён: {not_prepared.mean():.2f}")
print(f"Welch t-statistic: {t_stat:.3f}; p-value: {p_value:.6g}")

Использован двусторонний t-тест Уэлча, который не требует предполагать одинаковые дисперсии групп. Если p-value меньше 0.05, средние результаты по математике статистически различаются. По самим средним можно определить направление различия. Это наблюдательная связь: тест не доказывает, что курс подготовки сам по себе вызвал повышение результата.

### Exercise 2.3 — ANOVA across groups
Test whether mean `math score` differs across `parental level of education`
groups using a one-way ANOVA (`stats.f_oneway`). Interpret the p-value.


In [ ]:
education_groups = [group["math score"].to_numpy()
                    for _, group in students.groupby("parental level of education")]
f_stat, p_value = stats.f_oneway(*education_groups)
print("Средние math score по уровню образования родителей:")
print(students.groupby("parental level of education")["math score"].mean().sort_values().round(2))
print(f"\nANOVA F-statistic: {f_stat:.3f}; p-value: {p_value:.6g}")
if p_value < 0.05:
    print("Вывод: есть статистически значимые различия хотя бы между двумя средними групп.")
else:
    print("Вывод: статистически значимых различий средних не обнаружено.")

### Exercise 2.4 — Chi-square test of independence
Build a contingency table of `lunch` (standard/free-reduced) vs.
`test preparation course`, and run a chi-square test of independence
(`stats.chi2_contingency`). Are the two variables associated?


In [ ]:
contingency = pd.crosstab(students["lunch"], students["test preparation course"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print("Таблица сопряжённости:")
print(contingency)
print(f"\nChi-square: {chi2:.3f}; degrees of freedom: {dof}; p-value: {p_value:.6g}")
print("Ожидаемые частоты при независимости:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).round(1))
if p_value < 0.05:
    print("Вывод: lunch и test preparation course статистически связаны.")
else:
    print("Вывод: статистически значимой связи не обнаружено.")

## Ответы на вопросы

1. **p-value** — вероятность получить такие же или более экстремальные данные при условии, что нулевая гипотеза верна. Это не вероятность того, что H₀ верна, и не вероятность случайности результата.
2. Статистическая значимость не показывает размер эффекта. Малое различие может стать значимым при большой выборке, но быть практически несущественным.
3. Возможный смешивающий фактор: социально-экономическое положение семьи может влиять и на доступ к подготовке/типу обеда, и на результаты экзаменов. Наблюдаемая ассоциация сама по себе не устанавливает причинность.